In [1]:
import numpy as np
import pandas as pd

bars = pd.read_csv("data/processed/bars_clean.csv", parse_dates=["bar_start_utc"])
bars = bars.sort_values(["symbol", "bar_start_utc"]).reset_index(drop=True)
print(len(bars), "bars")

115033 bars


In [2]:
day = bars.groupby(["symbol", "session_date"]).agg(
    session_open=("open", "first"),
    session_close=("close", "last"),
    session_high=("high", "max"),
    session_low=("low", "min"),
    session_volume=("volume", "sum"),
    observed_bars=("close", "size"),
).reset_index()

print(len(day), "stock-days | fewest bars:", day["observed_bars"].min())

1475 stock-days | fewest bars: 76


In [3]:
day["day_direction"] = np.select(
    [day["session_close"] > day["session_open"], day["session_close"] < day["session_open"]],
    ["Up", "Down"],
    default="Flat",
)
day["day_direction"].value_counts()

day_direction
Down    838
Up      634
Flat      3
Name: count, dtype: int64

In [4]:
g = bars.groupby(["symbol", "session_date"])
prev_close = g["close"].shift(1)
back_to_back = g["bar_start_utc"].diff() == pd.Timedelta(minutes=5)

bars["log_ret"] = np.where(back_to_back, np.log(bars["close"] / prev_close), np.nan)
rv = (bars["log_ret"] ** 2).groupby([bars["symbol"], bars["session_date"]]).sum(min_count=1) ** 0.5
day = day.merge(rv.rename("realized_volatility").reset_index(), on=["symbol", "session_date"])

In [5]:
import json
config = json.load(open("config.json"))

is_calib = day["session_date"].isin(config["calibration_days"])
q1, q2 = day.loc[is_calib, "realized_volatility"].quantile([1/3, 2/3])

day["volatility_tier"] = pd.cut(day["realized_volatility"], bins=[-np.inf, q1, q2, np.inf],
                                labels=["Low", "Medium", "High"])
print("Cutoffs:", q1, q2)
day["volatility_tier"].value_counts(sort=False)

Cutoffs: 0.010121135659095459 0.013165860565478218


volatility_tier
Low       523
Medium    466
High      486
Name: count, dtype: int64

In [6]:
day.to_csv("data/processed/stock_day_all.csv", index=False)
print("Saved", len(day), "rows")

Saved 1475 rows


In [7]:
pd.read_csv("data/processed/stock_day_all.csv")

,symbol,session_date,session_open,session_close,session_high,session_low,session_volume,observed_bars,day_direction,realized_volatility,volatility_tier
0,AAPL,2026-08-14,305.940,305.980,307.4900,304.300,18872305.0,78,Up,0.007128,Low
1,AAPL,2026-08-17,306.000,305.690,307.6600,302.939,24373707.0,78,Down,0.007690,Low
2,AAPL,2026-08-18,307.580,310.210,311.4900,305.740,36650312.0,78,Up,0.010856,Medium
3,AAPL,2026-08-19,310.140,316.900,319.2799,309.600,36353411.0,78,Up,0.016465,High
4,AAPL,2026-08-20,317.455,311.295,320.2800,310.650,24621808.0,78,Down,0.011369,Medium
...,...,...,...,...,...,...,...,...,...,...,...
1470,XOM,2026-09-14,168.700,165.070,169.4500,164.040,10040084.0,78,Down,0.015757,High
1471,XOM,2026-09-15,165.360,169.290,169.6400,165.150,9705843.0,78,Up,0.013958,High
1472,XOM,2026-09-16,167.230,163.330,167.4950,162.560,9973469.0,78,Down,0.012708,Medium
1473,XOM,2026-09-17,161.830,163.260,163.3850,161.445,7637184.0,78,Up,0.009235,Low


In [8]:
calib = day[day["session_date"].isin(config["calibration_days"])]

prof = calib.groupby("symbol").agg(
    mean_daily_volume=("session_volume", "mean"),
    mean_close_usd=("session_close", "mean"),
    valid_calibration_days=("observed_bars", lambda b: (b >= 75).sum()),
).reset_index()
print(len(prof), "stocks")

59 stocks


In [9]:
l1, l2 = prof["mean_daily_volume"].quantile([1/3, 2/3])
prof["liquidity_tier"] = pd.cut(prof["mean_daily_volume"], bins=[-np.inf, l1, l2, np.inf],
                                labels=["Low", "Medium", "High"])
print("Liquidity cutoffs:", l1, l2)
prof["liquidity_tier"].value_counts(sort=False)

Liquidity cutoffs: 2298919.133333333 6147448.6


liquidity_tier
Low       20
Medium    19
High      20
Name: count, dtype: int64

In [11]:
meta = pd.read_csv("data/processed/stock_metadata.csv", dtype={"cik": str})

profile = meta.merge(prof, left_on="ticker", right_on="symbol").drop(columns="symbol")
profile = profile.rename(columns={"name": "company_name"})
profile["calibration_end_date"] = config["calibration_days"][-1]
print(len(profile), "rows")
profile.head()

59 rows


,cik,company_name,ticker,exchange,snapshot_date,mean_daily_volume,mean_close_usd,valid_calibration_days,liquidity_tier,calibration_end_date
0,0001045810,NVIDIA CORP,NVDA,Nasdaq,2026-09-24,70757435.8,220.9240,5,High,2026-08-20
1,0000320193,Apple Inc.,AAPL,Nasdaq,2026-09-24,28174308.6,310.0150,5,High,2026-08-20
2,0001652044,Alphabet Inc.,GOOGL,Nasdaq,2026-09-24,14124536.0,343.8860,5,High,2026-08-20
3,0000789019,MICROSOFT CORP,MSFT,Nasdaq,2026-09-24,14632889.8,484.7278,5,High,2026-08-20
4,0001018724,AMAZON COM INC,AMZN,Nasdaq,2026-09-24,26666928.4,261.9020,5,High,2026-08-20


In [12]:
config["volatility_cutoffs"] = [float(q1), float(q2)]
config["liquidity_cutoffs"] = [float(l1), float(l2)]
config["quantile_method"] = "pandas default (linear interpolation)"
with open("config.json", "w") as f:
    json.dump(config, f, indent=2)

profile.to_csv("data/processed/stock_profile.csv", index=False)
print("Saved profile and cutoffs")

Saved profile and cutoffs


In [ ]:
main_start = config["main_start"]

intraday = bars[bars["session_date"] >= main_start].copy()
intraday["bar_start_utc"] = intraday["bar_start_utc"].dt.tz_localize(None)
intraday = intraday.rename(columns={"symbol": "ticker"})[
    ["ticker", "feed", "bar_start_utc", "session_date", "session_stage",
     "open", "high", "low", "close", "volume", "trade_count", "vwap"]]

stock_day = day[day["session_date"] >= main_start].rename(columns={"symbol": "ticker"})

profile = profile[["ticker", "cik", "company_name", "exchange", "snapshot_date", "calibration_end_date",
                   "mean_daily_volume", "mean_close_usd", "valid_calibration_days", "liquidity_tier"]]

filename = "221953930-219431865.xlsx"
with pd.ExcelWriter(filename) as writer:
    profile.to_excel(writer, sheet_name="Stock_Profile", index=False)
    stock_day.to_excel(writer, sheet_name="Stock_Day", index=False)
    intraday.to_excel(writer, sheet_name="Intraday_Bars", index=False)
    for sheet in writer.sheets.values():
        for col in sheet.columns:
            sheet.column_dimensions[col[0].column_letter].width = 20

print(len(profile), len(stock_day), len(intraday))

59 1180 92026
